In [2]:
import arcpy
import os

def merge_feature_classes_from_multiple_gdb(input_gdb_list, output_gdb):
    """
    合并多个FileGDB中同名的要素类（包括要素数据集内的），并保留原有的要素数据集结构，
    最终生成一个新的FileGDB。

    参数:
    - input_gdb_list: 包含所有输入 File GDB 完整路径的列表。
    - output_gdb: 新建输出 File GDB 的完整路径。
    """
    
    # 1. 检查输出 GDB 的父文件夹是否存在
    output_gdb_dir = os.path.dirname(output_gdb)
    if not os.path.exists(output_gdb_dir):
        print(f"输出目录 '{output_gdb_dir}' 不存在，请先创建。")
        return

    # 2. 如果输出 GDB 已存在，则先删除
    if arcpy.Exists(output_gdb):
        print(f"输出 File GDB '{output_gdb}' 已存在，正在删除...")
        arcpy.Delete_management(output_gdb)
    
    # 3. 创建一个新的 File GDB
    arcpy.CreateFileGDB_management(output_gdb_dir, os.path.basename(output_gdb))
    print(f"已创建新的 File GDB: '{output_gdb}'")
    print("-" * 50)

    # 4. 获取所有输入 GDB 中的要素数据集和根目录要素类
    gdb_contents = {}
    for gdb_path in input_gdb_list:
        if not arcpy.Exists(gdb_path):
            print(f"警告: 找不到输入 GDB '{gdb_path}'，已跳过。")
            continue
        
        arcpy.env.workspace = gdb_path
        
        # 获取根目录下的要素类
        fcs = arcpy.ListFeatureClasses()
        # 获取要素数据集
        datasets = arcpy.ListDatasets(feature_type='Feature')
        
        gdb_contents[gdb_path] = {'root_fcs': fcs, 'datasets': {}}
        for dataset in datasets:
            arcpy.env.workspace = os.path.join(gdb_path, dataset)
            gdb_contents[gdb_path]['datasets'][dataset] = arcpy.ListFeatureClasses()

    # 恢复工作空间为初始状态
    arcpy.env.workspace = output_gdb_dir
    
    # 5. 合并根目录下的要素类
    print("开始合并根目录下的要素类...")
    
    # 获取所有 GDB 中根目录要素类的唯一名称
    all_root_fcs = set()
    for gdb in gdb_contents:
        all_root_fcs.update(gdb_contents[gdb]['root_fcs'])
    
    for fc_name in all_root_fcs:
        # 收集所有同名要素类的完整路径
        input_fcs = [os.path.join(gdb, fc_name) for gdb in gdb_contents if fc_name in gdb_contents[gdb]['root_fcs']]
        
        if len(input_fcs) > 0:
            output_fc = os.path.join(output_gdb, fc_name)
            
            print(f"  - 正在合并要素类 '{fc_name}'，共找到 {len(input_fcs)} 个源。")
            
            # 合并同名要素类
            try:
                arcpy.Merge_management(input_fcs, output_fc)
                print(f"  - 成功合并到 '{output_fc}'。")
            except Exception as e:
                print(f"  - 合并 '{fc_name}' 时出错: {e}")
    
    print("-" * 50)
    
    # 6. 合并要素数据集及其中的要素类
    print("开始合并要素数据集及其中的要素类...")

    # 获取所有 GDB 中要素数据集的唯一名称
    all_datasets = set()
    for gdb in gdb_contents:
        all_datasets.update(gdb_contents[gdb]['datasets'].keys())
    
    for dataset_name in all_datasets:
        # 在输出 GDB 中创建同名的要素数据集
        output_dataset = os.path.join(output_gdb, dataset_name)
        
        # 获取第一个包含该数据集的 GDB，以获取空间参考
        first_gdb_with_dataset = next((gdb for gdb, content in gdb_contents.items() if dataset_name in content['datasets']), None)
        if first_gdb_with_dataset:
            # 获取该数据集的空间参考
            first_dataset_path = os.path.join(first_gdb_with_dataset, dataset_name)
            spatial_ref = arcpy.Describe(first_dataset_path).spatialReference
            
            arcpy.CreateFeatureDataset_management(output_gdb, dataset_name, spatial_ref)
            print(f"  - 已在输出 GDB 中创建要素数据集 '{dataset_name}'。")

            # 获取所有 GDB 中该要素数据集内的要素类名称
            all_dataset_fcs = set()
            for gdb in gdb_contents:
                if dataset_name in gdb_contents[gdb]['datasets']:
                    all_dataset_fcs.update(gdb_contents[gdb]['datasets'][dataset_name])

            # 遍历并合并每个要素数据集内的要素类
            for fc_name in all_dataset_fcs:
                input_fcs = []
                for gdb, content in gdb_contents.items():
                    if dataset_name in content['datasets'] and fc_name in content['datasets'][dataset_name]:
                        input_fcs.append(os.path.join(gdb, dataset_name, fc_name))
                
                if len(input_fcs) > 0:
                    output_fc = os.path.join(output_dataset, fc_name)
                    
                    print(f"  - 正在合并要素数据集 '{dataset_name}' 下的要素类 '{fc_name}'，共找到 {len(input_fcs)} 个源。")
                    
                    try:
                        arcpy.Merge_management(input_fcs, output_fc)
                        print(f"  - 成功合并到 '{output_fc}'。")
                    except Exception as e:
                        print(f"  - 合并 '{fc_name}' 时出错: {e}")
    
    print("-" * 50)
    print("所有要素类合并完成。")


if __name__ == '__main__':
    # ----------------------------------------------------
    # 用户需要修改以下参数
    # ----------------------------------------------------

    # 1. 指定所有输入 File GDB 的完整路径
    # 例如: gdb1.gdb, gdb2.gdb
    input_gdb_paths = [
        r"E:\成果库\市级成果251222\市级成果251222\510104锦江区_CDCSJC2025.gdb",
        r"E:\成果库\市级成果251222\市级成果251222\510105青羊区_CDCSJC2025.gdb",
        r"E:\成果库\市级成果251222\市级成果251222\510106金牛区_CDCSJC2025.gdb",
        r"E:\成果库\市级成果251222\市级成果251222\510107武侯区_CDCSJC2025.gdb",
        r"E:\成果库\市级成果251222\市级成果251222\510108成华区_CDCSJC2025.gdb",
        r"E:\成果库\市级成果251222\市级成果251222\510109高新区_CDCSJC2025.gdb",
        r"E:\成果库\市级成果251222\市级成果251222\510110天府新区成都直管区_CDCSJC2025.gdb",
        r"E:\成果库\市级成果251222\市级成果251222\510112龙泉驿区_CDCSJC2025.gdb",
        r"E:\成果库\市级成果251222\市级成果251222\510113青白江区_CDCSJC2025.gdb",
        r"E:\成果库\市级成果251222\市级成果251222\510114新都区_CDCSJC2025.gdb",
        r"E:\成果库\市级成果251222\市级成果251222\510115温江区_CDCSJC2025.gdb",
        r"E:\成果库\市级成果251222\市级成果251222\510116双流区_CDCSJC2025.gdb",
        r"E:\成果库\市级成果251222\市级成果251222\510117郫都区_CDCSJC2025.gdb",
        r"E:\成果库\市级成果251222\市级成果251222\510118新津区_CDCSJC2025.gdb",
        r"E:\成果库\市级成果251222\市级成果251222\510121金堂县_CDCSJC2025.gdb",
        r"E:\成果库\市级成果251222\市级成果251222\510129大邑县_CDCSJC2025.gdb",
        r"E:\成果库\市级成果251222\市级成果251222\510131蒲江县_CDCSJC2025.gdb",
        r"E:\成果库\市级成果251222\市级成果251222\510181都江堰市_CDCSJC2025.gdb",
        r"E:\成果库\市级成果251222\市级成果251222\510182彭州市_CDCSJC2025.gdb",
        r"E:\成果库\市级成果251222\市级成果251222\510183邛崃市_CDCSJC2025.gdb",
        r"E:\成果库\市级成果251222\市级成果251222\510184崇州市_CDCSJC2025.gdb",
        r"E:\成果库\市级成果251222\市级成果251222\510185简阳市_CDCSJC2025.gdb"
    ]

    # 2. 指定新创建的输出 File GDB 的完整路径
    output_gdb_path = r"E:\成果库\CDData1222.gdb"

    # ----------------------------------------------------
    # 调用函数，开始执行脚本
    # ----------------------------------------------------
    merge_feature_classes_from_multiple_gdb(input_gdb_paths, output_gdb_path)

已创建新的 File GDB: 'E:\成果库\CDData1222.gdb'
--------------------------------------------------
开始合并根目录下的要素类...
  - 正在合并要素类 'CDZZJCFWA'，共找到 22 个源。
  - 成功合并到 'E:\成果库\CDData1222.gdb\CDZZJCFWA'。
--------------------------------------------------
开始合并要素数据集及其中的要素类...
  - 已在输出 GDB 中创建要素数据集 'BCDataset'。
  - 正在合并要素数据集 'BCDataset' 下的要素类 'HYDL'，共找到 22 个源。
  - 成功合并到 'E:\成果库\CDData1222.gdb\BCDataset\HYDL'。
  - 正在合并要素数据集 'BCDataset' 下的要素类 'DXKJA'，共找到 22 个源。
  - 成功合并到 'E:\成果库\CDData1222.gdb\BCDataset\DXKJA'。
  - 正在合并要素数据集 'BCDataset' 下的要素类 'CQNFWJZA'，共找到 22 个源。
  - 成功合并到 'E:\成果库\CDData1222.gdb\BCDataset\CQNFWJZA'。
  - 正在合并要素数据集 'BCDataset' 下的要素类 'SWHXCBCTQTBA'，共找到 22 个源。
  - 成功合并到 'E:\成果库\CDData1222.gdb\BCDataset\SWHXCBCTQTBA'。
  - 正在合并要素数据集 'BCDataset' 下的要素类 'XZCSGXGZYDA'，共找到 22 个源。
  - 成功合并到 'E:\成果库\CDData1222.gdb\BCDataset\XZCSGXGZYDA'。
  - 正在合并要素数据集 'BCDataset' 下的要素类 'LCTL'，共找到 22 个源。
  - 成功合并到 'E:\成果库\CDData1222.gdb\BCDataset\LCTL'。
  - 正在合并要素数据集 'BCDataset' 下的要素类 'DXKJL'，共找到 22 个源。
  - 成功合并到 'E:\成果